In [1]:
import re
import os
import glob
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from collections import defaultdict
from cartopy.util import add_cyclic_point

In [2]:
def dpres_hybrid_ccm(PS, P0, hyai, hybi):
    """
    Python replacement for NCL's dpres_hybrid_ccm
    PS   : (ens, time, lat, lon)
    P0   : (ens,)
    hyai : (ens, nlev+1,)
    hybi : (ens, nlev+1,)
    """
    # Expand dimensions for broadcasting
    hyai_exp = hyai[:, :, None, None, None]   # (ens, lev+1, 1, 1, 1)
    hybi_exp = hybi[:, :, None, None, None]   # (ens, lev+1, 1, 1, 1)
    P0_exp = P0[:, None, None, None, None]    # (ens, 1, 1, 1, 1)
    PS_exp = PS[:, None, :, :, :]             # (ens, 1, time, lat, lon)

    # Compute interface pressures
    pint = hyai_exp * P0_exp + hybi_exp * PS_exp  # (ens, lev+1, time, lat, lon)

    # dp: (lev, time, lat, lon)
    dp = pint[:, 1:, :, :, :] - pint[:, :-1, :, :, :]
    return dp


def prcwater_dp(q, dp, g=9.80665):
    """
    Python replacement for NCL's prcwater_dp
    q  : specific humidity  (ncol, nlev)
    dp : pressure thickness (ncol, nlev)
    """
    return (q * dp).sum(axis=2) / g

## Loading data into dataframe

**Variable Names:**
- CLOUD: cloud fraction
- FICE: cloud type
- FSDSC: downwelling shortwave clear sky
- FSDS: downwelling shortwave
- FSNSC: net shortwave clear sky
- FSNS: net shortwave
- PRECC: convective precipitation rate (liquid + ice)
- PRECL: large scale precipitation rate (liquid + ice)
- PRECSC: convective snow rate
- PRECSL: large scale snow rate
- PSL: sea level pressure
- Q: specific humidity
- TREFHT: 2 meter temperature
- WPRTP_CLUBB: moisture flux

**NCL Functions:**
- [dpres_hybrid_cmm](https://www.ncl.ucar.edu/Document/Functions/Built-in/dpres_hybrid_ccm.shtml): Calculates the pressure layer thicknesses of a hybrid coordinate system.
- [prcwater_dp](https://www.ncl.ucar.edu/Document/Functions/Built-in/prcwater_dp.shtml): Computes total column precipitable water of the rightmost dimension.

These functions are rewritten for Python (above) since they were not transferred from the original NCL library (C++/Fortran) to the [geocat-comp](https://geocat-comp.readthedocs.io/en/stable/getting_started/quick-install.html) library installed on this server.

In [3]:
# Get all files
paths = sorted(glob.glob('/glade/derecho/scratch/skygale/nudge_moisture_budget_vars/PInudge/*195001-202312*.nc'))

# Group files by variable name
file_dict = defaultdict(list)
for path in paths:
    fname = os.path.basename(path)
    var_match = re.search(r'(CLOUD|FICE|FSDSC|FSDS|FSNSC|FSNS|PRECC|PRECL|PRECSC|PRECSL|PSL|Q|TREFHT|WPRTP_CLUBB)', fname)
    if var_match:
        varname = var_match.group(1)
        file_dict[varname].append(path)

# Put all processed data together
arrays = []
for varname, files in file_dict.items():
    print(f"Processing {varname} from {len(files)} file(s)...")

    # Each file is a different ensemble member
    datasets = []
    for path in files:
        ens_match = re.search(r'\.(\d{3})\.h0', os.path.basename(path))
        ens = ens_match.group(1) if ens_match else '000'
        ds = xr.open_dataset(path, chunks={'time': 12})
        ds = ds.assign_coords(ensemble=int(ens))
        datasets.append(ds)

    # Combine along a new ensemble dimension
    ds = xr.concat(datasets, dim='ensemble')

    # Select March–May, 20–90N, 1980–2022
    ds = ds.sel(time=slice("1980-01-01", "2022-12-31"), lat=slice(20, 90))
    ds = ds.sel(time=ds['time'].dt.month.isin([4, 5, 6]))

    # Select variable
    da = ds[varname]

    # Handle pressure level variables
    if varname in ['CLOUD', 'FICE']:
        # Pressure-weighted mean over lev
        plev = da.lev
        delta_p = np.diff(plev)
        delta_p = np.append(delta_p, delta_p[-1])
        delta_p = xr.DataArray(delta_p, dims='lev', coords={'lev': da.lev})

        dx = ((da * delta_p).sum(dim='lev') / delta_p.sum())
        dx = dx.expand_dims({'variable': [varname]})

    elif varname == 'Q':
        # Integrated water vapor (IWV)
        Q = da.values
        PS = ds.PS.values
        P0 = ds.P0.values
        hyai = ds.hyai.values
        hybi = ds.hybi.values

        # Calculate layer thickness (dp) using dpres_hybrid_ccm
        dp = dpres_hybrid_ccm(PS, P0, hyai, hybi)

        # Transpose to (ens, time, lev, lat, lon)
        dp = np.transpose(dp, (0, 2, 1, 3, 4))

        # Calculate IWV, weighed by the thickness of each layer, from Q using prcwater_dp
        dx = prcwater_dp(Q, dp)

        # Wrap as xarray
        dx_da = xr.DataArray(
            dx,
            dims=["ensemble", "time", "lat", "lon"],
            coords={
                "ensemble": ds.ensemble,
                "time": ds.time,
                "lat": ds.lat,
                "lon": ds.lon,
            }
        )

        dx = dx_da.expand_dims({'variable': ['IWV']})

    elif varname == 'WPRTP_CLUBB':
        # Pressure-weighted mean over ilev
        plev = da.ilev
        delta_p = np.diff(plev)
        delta_p = np.append(delta_p, delta_p[-1])
        delta_p = xr.DataArray(delta_p, dims='ilev', coords={'ilev': da.ilev})

        dx = ((da * delta_p).sum(dim='ilev') / delta_p.sum())
        dx = dx.expand_dims({'variable': ['MOISTUREFLUX']})

    else:
        dx = da.expand_dims({'variable': [varname]})

    arrays.append(dx)

# Combine all variables
ds = xr.concat(arrays, dim='variable')

# Ensemble mean
ds = ds.mean(dim='ensemble')
ds

Processing CLOUD from 3 file(s)...
Processing FICE from 3 file(s)...
Processing FSDS from 3 file(s)...
Processing FSDSC from 3 file(s)...
Processing FSNS from 3 file(s)...
Processing FSNSC from 3 file(s)...
Processing PRECC from 3 file(s)...
Processing PRECL from 3 file(s)...
Processing PRECSC from 3 file(s)...
Processing PRECSL from 3 file(s)...
Processing PSL from 3 file(s)...
Processing Q from 3 file(s)...
Processing TREFHT from 3 file(s)...
Processing WPRTP_CLUBB from 3 file(s)...


<xarray.DataArray (variable: 14, time: 129, lat: 75, lon: 288)> Size: 312MB
dask.array<mean_agg-aggregate, shape=(14, 129, 75, 288), dtype=float64, chunksize=(1, 3, 75, 288), chunktype=numpy.ndarray>
Coordinates:
  * variable  (variable) object 112B 'CLOUD' 'FICE' ... 'TREFHT' 'MOISTUREFLUX'
  * lat       (lat) float64 600B 20.26 21.2 22.15 23.09 ... 88.12 89.06 90.0
  * lon       (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 355.0 356.2 357.5 358.8
  * time      (time) object 1kB 1980-04-01 00:00:00 ... 2022-06-01 00:00:00

In [4]:
# Create year and month coordinates
years = np.unique(ds['time'].dt.year)
months = np.unique(ds['time'].dt.month)

# Group by year and month using time.dt
df = ds.groupby('time.year').apply(
    lambda ds_year: ds_year.groupby('time.month').mean('time')
)

# Now df has dimensions ('variable', 'year', 'month', 'lat', 'lon')
# But sometimes 'year' and 'month' are coordinates, not dims. Let's fix that:
df = df.assign_coords(
    year=years,
    month=months
).transpose('variable', 'year', 'month', 'lat', 'lon')

df.name = 'nudged_variables'
df.dims

('variable', 'year', 'month', 'lat', 'lon')

## Calculate trends of variables

In [5]:
# Compute the linear trend (slope) for each variable, month, lat, lon
trend = df.polyfit(dim='year', deg=1)['polyfit_coefficients'].sel(degree=1) * 10
trend.dims

('variable', 'month', 'lat', 'lon')

## Plotting trends

In [6]:
# Define projections
proj = ccrs.NorthPolarStereo()
data_crs = ccrs.PlateCarree()

lats = trend.lat.values
lons = trend.lon.values

# Define month labels
month_labels = {4: "March", 5: "April", 6: "May"}

# Define variable qualities
vars_long = [
    'Cloud Fraction',
    'Fractional Ice Content within Cloud',
    'Downwelling Shortwave Flux at Surface',
    'Clearsky Downwelling Shortwave Flux at Surface',
    'Net Shortwave Flux at Surface',
    'Clearsky Net Shortwave Flux at Surface',
    'Convective Precipitation Rate (liquid + ice)',
    'Large Scale Precipitation Rate (liquid + ice)',
    'Convective Snow Rate (Water Equivalent)',
    'Large Scale (Stable) Snow Rate (Water Equivalent)',
    'Sea Level Pressure',
    'Total Column Water Vapor',
    '2m Surface Air Temperature',
    'Moisture Flux',
]

vars_units = [
    'Fraction',
    'Fraction',
    'W m⁻²',
    'W m⁻²',
    'W m⁻²',
    'W m⁻²',
    'm s⁻¹',
    'm s⁻¹',
    'm s⁻¹',
    'm s⁻¹',
    'Pa',
    'kg m⁻²',
    'K',
    'W m⁻²'
]

print("Loaded trend:", trend.shape, flush=True)

Loaded trend: (14, 3, 75, 288)


In [ ]:
# Iterate over variables
for v, var in enumerate(trend['variable'].values):
    print(f"Variable {v+1}/{len(trend['variable'])}: {var}", flush=True)

    fig, axes = plt.subplots(1, 3, figsize=(8, 4), constrained_layout=True, subplot_kw={'projection': proj})

    # Extract current variable
    tvar = trend.sel(variable=var)

    # Get color range for consistent scaling
    absmax = np.max(abs(tvar))

    for i, (month, ax) in enumerate(zip(month_labels.keys(), axes)):
        print(f"  Month {month}", flush=True)

        ax.set_extent([-180, 180, 50, 90], crs=data_crs)
        ax.gridlines(linestyle=':', alpha=0.3)
        ax.coastlines(resolution='110m', linewidth=0.5)

        da_plot = tvar.sel(month=month)

        # Make cyclic
        da_cyclic, lons_cyclic = add_cyclic_point(da_plot, coord=lons)

        im = ax.contourf(lons_cyclic, lats, da_cyclic,
                         transform=data_crs, cmap='RdBu_r',
                         levels=np.linspace(-absmax, absmax, 20), extend='both',
                        )

        ax.set_title(f"{month_labels[month]}")

    # Add colorbar under subplots
    if im is not None:
        cbar = fig.colorbar(
            im, ax=axes, orientation='horizontal',
            aspect=30, shrink=0.6,
            ticks=np.linspace(-absmax, absmax, 5)
        )
        cbar.set_label(vars_units[v] + ' decade⁻¹', fontsize=11)

    plt.suptitle(vars_long[v], fontsize=14)
    plt.savefig(f'/glade/derecho/scratch/skygale/nudged_plots/{vars_long[v]}.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)

In [ ]:
# Select variable
var = "IWV"

# Ensure values are plain Python strings
vars_list = trend.variable.values.astype(str).tolist()
v = vars_list.index(var)
print(f"Plotting only variable '{var}'")

fig, axes = plt.subplots(1, 3, figsize=(8, 4), constrained_layout=True, subplot_kw={'projection': proj})

# Extract WV variable
tvar = trend.sel(variable=var)

# Get color range for consistent scaling
absmax = np.max(abs(tvar))

for i, (month, ax) in enumerate(zip(month_labels.keys(), axes)):
    print(f"  Month {month}", flush=True)

    ax.set_extent([-180, 180, 50, 90], crs=data_crs)
    ax.gridlines(linestyle=':', alpha=0.3)
    ax.coastlines(resolution='110m', linewidth=0.5)

    da_plot = tvar.sel(month=month)

    # Make cyclic
    da_cyclic, lons_cyclic = add_cyclic_point(da_plot, coord=lons)

    im = ax.contourf(lons_cyclic, lats, da_cyclic,
                     transform=data_crs, cmap='RdBu_r',
                     levels=np.linspace(-absmax, absmax, 20), extend='both'
                    )

    ax.set_title(f"{month_labels[month]}")

# Add colorbar under subplots
cbar = fig.colorbar(im, ax=axes, orientation='horizontal',
                    aspect=30, shrink=0.6,
                    ticks=np.linspace(-absmax, absmax, 5))
cbar.set_label(vars_units[v] + ' decade⁻¹', fontsize=11)

plt.suptitle(vars_long[v], fontsize=14)
plt.savefig(f'/glade/derecho/scratch/skygale/nudged_plots/{vars_long[v]}.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)